# Evaluating Amazon Bedrock AgentCore Memory with EvalKit memory metrics

## 1. Overview

**Question:** does AgentCore Memory let an assistant remember a user across sessions about as well as giving it the user's whole history?

This cookbook walks through the full evaluation, end to end:

1. Build ground-truth test data: a synthetic user whose facts change over time.
2. Create an **AgentCore Memory** store with three long-term strategies and ingest the history as backdated events.
3. Run a **Strands agent** (the kind you would host on AgentCore Runtime) that reads memory through a top-K index and a `search_memory` tool.
4. Turn the answers and memory snapshot into **EvalKit** rows and score them with EvalKit's ten managed memory evaluators: seven code metrics, which this notebook re-implements locally, and three LLM judges.

![Memory Eval with Syntropylabs.ai: the evaluation workflow](images/eval_workflow.png)

*Figure: Memory Eval with Syntropylabs.ai, the workflow this notebook follows.*

### Design: memory vs a full-context ceiling

Every question is asked in a **brand-new session**, which simulates context eviction. The same agent and model answer under two conditions:

| | **Memory** (system under test) | **Full context** (ceiling) |
|---|---|---|
| What the agent receives | The question, the top-5 memory records as a `<memory>` block, and a `search_memory` tool | The question and the user's **entire past transcript** |
| Writes during evaluation | Off (`read_only`), so one question can't leak into the next | n/a |
| What it costs | A few thousand input tokens | Grows with the history |

The gap between the two conditions shows how much memory loses compared with reading everything.

## 2. Setup

The notebook runs **offline by default**: every cell that would call AWS or EvalKit is guarded by `RUN_LIVE` and only prints what it would do. Offline cells use the precomputed results in `sample_data/`, which come from a full run (`full-2`). Set `RUN_LIVE = True` and the environment variables below to run against your own account. That creates AWS resources and incurs cost.

In [1]:
RUN_LIVE = False  # True = create AWS resources and call Bedrock / EvalKit (costs money)

import json
import os
from pathlib import Path

from IPython.display import Markdown, display

DATA = Path("sample_data")

# Configuration comes from environment variables (names only; never hard-code values).
ENV_VARS = [
    "AWS_REGION",                # e.g. us-west-2
    "MEMORY_ID",                 # set after section 4 creates the memory
    "MODEL_ID",                  # Bedrock model or inference profile id for the agent
    "EVALKIT_SUBSCRIPTION_KEY",  # EvalKit SDK key (secret)
    "EVALKIT_COLLECTION_ID",     # optional: an EvalKit evaluator collection
    "EVALKIT_ENVIRONMENT",       # e.g. development
    "EVALKIT_JUDGE_MODEL",       # optional: judge model for EvalKit's LLM evaluators
]
REGION = os.environ.get("AWS_REGION", "us-west-2")
MODEL_ID = os.environ.get("MODEL_ID", "global.anthropic.claude-sonnet-5-5")
K = 5  # memories injected into the prompt, and the K in Precision@K / Recall@K


def offline(what: str) -> None:
    print(f"[offline] would {what}")


def table(rows, cols):
    """Render a list of dicts as a Markdown table."""
    esc = lambda v: str(v).replace("|", "\\|").replace("\n", " ")
    lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    lines += ["| " + " | ".join(esc(r.get(c, "")) for c in cols) + " |" for r in rows]
    display(Markdown("\n".join(lines)))


print("RUN_LIVE =", RUN_LIVE)
print({name: ("set" if os.environ.get(name) else "unset") for name in ENV_VARS})

RUN_LIVE = False
{'AWS_REGION': 'unset', 'MEMORY_ID': 'unset', 'MODEL_ID': 'unset', 'EVALKIT_SUBSCRIPTION_KEY': 'unset', 'EVALKIT_COLLECTION_ID': 'unset', 'EVALKIT_ENVIRONMENT': 'unset', 'EVALKIT_JUDGE_MODEL': 'unset'}


## 3. Test data: timeline → sessions → questions

The test user is **Dara Kowalczyk**, an invented site reliability engineer (no real person or PII). The data is built in three layers:

- a **fact timeline**: dated facts, some of which **supersede** older ones (the cutover date moves April 18 → May 30 → July 18), plus **decoys**, which are plausible facts that are never stated;
- **sessions**: natural user ↔ assistant chats generated from the timeline, each tagged with the fact ids it states;
- **questions**, whose gold answers come **from the timeline**, not from an LLM's reading of the chats. Abstention questions ask about decoys, so the right answer is "I don't have that information."

The sample is trimmed to 7 of the persona's 25 sessions and 11 of its 40 questions.

In [2]:
persona = json.loads((DATA / "sre_persona_sample.json").read_text())
print(persona["persona"]["name"], "-", persona["persona"]["summary"][:110], "...")
print(f'{len(persona["facts"])} facts, {len(persona["sessions"])} sessions, '
      f'{sum(len(s["turns"]) for s in persona["sessions"])} turns, {len(persona["questions"])} questions')
table(persona["facts"], ["id", "stated_at", "memory_type", "value", "supersedes"])

Dara Kowalczyk - Site reliability engineer at Quillfield Payments, a mid-size payments company. Runs on-call, incident reviews  ...
14 facts, 7 sessions, 83 turns, 11 questions


| id | stated_at | memory_type | value | supersedes |
|---|---|---|---|---|
| f01 | 2026-01-05 | profile | Quillfield Payments | None |
| f02 | 2026-01-05 | profile | Brindlemoor | None |
| f03 | 2026-01-15 | projects | Project Heron | None |
| f04 | 2026-01-15 | projects | April 18 | None |
| f07 | 2026-02-12 | preferences | bullet lists | None |
| f08 | 2026-02-12 | domain_facts | PagerDuty | None |
| f09 | 2026-02-19 | workflows |  | None |
| f10 | 2026-02-28 | domain_facts | Oskar Brandt | None |
| f11 | 2026-02-28 | preferences | metric units | None |
| f12 | 2026-02-28 | profile | Biscuit | None |
| f13 | 2026-03-09 | projects | May 30 | f04 |
| f14 | 2026-03-09 | domain_facts | Granite Yard | None |
| f24 | 2026-05-22 | projects | July 18 | f13 |
| f25 | 2026-05-22 | domain_facts | 99.95 percent | None |

In [3]:
s = persona["sessions"][0]
print(s["session_id"], s["started_at"])
for t in s["turns"][:3]:
    print(f'  {t["role"]:>9}: {t["text"][:120]}  {t["fact_ids"] or ""}')

syn-sre-s01 2026-01-05T12:10:00+00:00
       user: need a short intro msg for a new slack channel. its for incident postmortem follow-ups. i'm an SRE at Quillfield Payment  ['f01']
  assistant: Draft:

"Welcome to #postmortem-followups. This channel tracks action items from incident reviews: owners, due dates, st  
       user: decent. drop the blameless line, that's already in the doc template. and add that items older than 30 days get flagged  


In [4]:
table(persona["questions"], ["qid", "category", "text", "expected_span", "fact_ids", "decoy_ids", "as_of"])

| qid | category | text | expected_span | fact_ids | decoy_ids | as_of |
|---|---|---|---|---|---|---|
| syn-sre-q01 | single_fact | Where do I work as a site reliability engineer? | Quillfield Payments | ['f01'] | [] | None |
| syn-sre-q02 | single_fact | What is my database migration project called? | Project Heron | ['f03'] | [] | None |
| syn-sre-q04 | single_fact | What's the name of my cat, and how old is she? | Biscuit | ['f12'] | [] | None |
| syn-sre-q05 | single_fact | Which bouldering gym do I go to? | Granite Yard | ['f14'] | [] | None |
| syn-sre-q07 | single_fact | What availability SLO does my payment API have? | 99.95 percent | ['f25'] | [] | None |
| syn-sre-q20 | temporal | Back in February, when was my Project Heron cutover scheduled? | April 18 | ['f04'] | [] | 2026-02-10 |
| syn-sre-q24 | latest_value | When is my Project Heron cutover currently scheduled? | July 18 | ['f24'] | [] | None |
| syn-sre-q30 | procedural | What are my steps when a production incident is declared? | incident commander | ['f09'] | [] | None |
| syn-sre-q31 | procedural | What do I do right after naming the incident commander during an incident? | every thirty minutes | ['f09'] | [] | None |
| syn-sre-q34 | abstention | What's my sister's name and where does she live? |  | [] | ['d01'] | None |
| syn-sre-q35 | abstention | What color is my car? |  | [] | ['d02'] | None |

## 4. Create AgentCore Memory with three strategies

The memory uses three built-in long-term strategies. Each writes to an **actor-scoped namespace**, so every read and write can be limited to one user:

| Strategy | Extracts | Namespace |
|---|---|---|
| Semantic | facts the user states | `/u/{actorId}/facts` |
| User preference | likes, dislikes, formats | `/u/{actorId}/prefs` |
| Summary | one summary per session | `/u/{actorId}/{sessionId}/summary` |

Extraction runs **asynchronously** after events arrive. `create_memory_and_wait` blocks until the memory is `ACTIVE`, which takes a few minutes.

In [5]:
STRATEGIES = [
    {"semanticMemoryStrategy": {"name": "Facts", "namespaces": ["/u/{actorId}/facts"]}},
    {"userPreferenceMemoryStrategy": {"name": "Preferences", "namespaces": ["/u/{actorId}/prefs"]}},
    {"summaryMemoryStrategy": {"name": "SessionSummary", "namespaces": ["/u/{actorId}/{sessionId}/summary"]}},
]


def create_memory(name: str = "memory_eval_cookbook") -> str:
    from bedrock_agentcore.memory import MemoryClient

    memory = MemoryClient(region_name=REGION).create_memory_and_wait(
        name=name,
        description="Semantic, user preference and summary strategies",
        strategies=STRATEGIES,
        event_expiry_days=90,
        max_wait=600,
        poll_interval=10,
    )
    return memory["id"]


if RUN_LIVE:
    MEMORY_ID = os.environ.get("MEMORY_ID") or create_memory()
    print("memory ready (export it as MEMORY_ID to reuse it)")
else:
    MEMORY_ID = "<MEMORY_ID>"
    offline(f"create a memory with {len(STRATEGIES)} strategies in {REGION}")

[offline] would create a memory with 3 strategies in us-west-2


## 5. Ingest history as backdated events, then snapshot the records

Each history turn becomes one `CreateEvent` call carrying the session's **original timestamp**. This keeps the timeline's order, so a strategy can tell which value is newer. After ingestion, poll `ListMemoryRecords` until the record count stops changing, then freeze a snapshot: the evaluation scores what memory actually holds.

These helpers call raw boto3 (`bedrock-agentcore` data plane) so record **ids and scores** are preserved, because the retrieval metrics need them.

In [6]:
import time
from datetime import datetime, timedelta

ROLES = {"user": "USER", "assistant": "ASSISTANT"}


def ingest_session(client, memory_id, actor_id, session):
    """One event per turn at the session's original time (+1 ms per turn keeps the order)."""
    start = datetime.fromisoformat(session["started_at"])
    for i, turn in enumerate(session["turns"]):
        client.create_event(
            memoryId=memory_id, actorId=actor_id, sessionId=session["session_id"],
            eventTimestamp=start + timedelta(milliseconds=i),
            payload=[{"conversational": {"content": {"text": turn["text"]}, "role": ROLES[turn["role"]]}}],
        )


def list_records(client, memory_id, actor_id):
    """Every long-term record under /u/{actor}/ (facts, prefs and all session summaries)."""
    kwargs = {"memoryId": memory_id, "namespacePath": f"/u/{actor_id}/", "maxResults": 100}
    out = []
    while True:
        page = client.list_memory_records(**kwargs)
        out += page.get("memoryRecordSummaries", [])
        if not page.get("nextToken"):
            return out
        kwargs["nextToken"] = page["nextToken"]


def wait_for_extraction(client, memory_id, actor_id, stable_polls=3, interval_s=10, timeout_s=600):
    """Poll until the record count is non-zero and unchanged for `stable_polls` polls."""
    last, streak, t0 = None, 0, time.monotonic()
    while time.monotonic() - t0 < timeout_s:
        n = len(list_records(client, memory_id, actor_id))
        streak, last = (streak + 1 if n == last else 1), n
        if n and streak >= stable_polls:
            return n
        time.sleep(interval_s)
    raise TimeoutError(f"extraction not stable after {timeout_s}s (last count {last})")


def search(client, memory_id, actor_id, query, k=K):
    """Top-k over the actor's facts and prefs namespaces, ranked by score: [(id, score, text)]."""
    best = {}
    for ns in (f"/u/{actor_id}/facts", f"/u/{actor_id}/prefs"):
        resp = client.retrieve_memory_records(
            memoryId=memory_id, namespace=ns, searchCriteria={"searchQuery": query, "topK": k})
        for r in resp.get("memoryRecordSummaries", []):
            hit = (r["memoryRecordId"], float(r.get("score") or 0.0), r["content"]["text"])
            if hit[0] not in best or hit[1] > best[hit[0]][1]:
                best[hit[0]] = hit
    return sorted(best.values(), key=lambda h: -h[1])[:k]

In [7]:
ACTOR = "cookbook-sre"  # one actor per test user; the full-context twin needs no memory

if RUN_LIVE:
    import boto3

    data_plane = boto3.client("bedrock-agentcore", region_name=REGION)
    for session in persona["sessions"]:
        ingest_session(data_plane, MEMORY_ID, ACTOR, session)
    print("records after extraction:", wait_for_extraction(data_plane, MEMORY_ID, ACTOR))
    snapshot = list_records(data_plane, MEMORY_ID, ACTOR)
    for rid, score, text in search(data_plane, MEMORY_ID, ACTOR, persona["questions"][6]["text"]):
        print(f"{score:.3f}  {text[:100]}")
else:
    n = sum(len(s["turns"]) for s in persona["sessions"])
    offline(f"write {n} events for actor {ACTOR!r}, "
            f"from {persona['sessions'][0]['started_at'][:10]} to {persona['sessions'][-1]['started_at'][:10]}")
    precomputed = json.loads((DATA / "full2_sre_rows.json").read_text())
    ex = precomputed["example_answers"]["syn-sre-q24"]
    print("\nPrecomputed top-5 search for:", ex["question"])
    for h in ex["memory"]["retrieved"]:
        print(f'  {h["score"]:.3f}  [{h["record_id"]}] {h["text"][:95]}')

[offline] would write 83 events for actor 'cookbook-sre', from 2026-01-05 to 2026-05-22

Precomputed top-5 search for: When is my Project Heron cutover currently scheduled?
  0.817  [rec-013] Project Heron cutover date was originally April 18, 2026, then switched to July 18, 2026.
  0.803  [rec-014] Project Heron has a cutover date of April 18, 2026. Pre-cutover gate process includes: (1) snap
  0.685  [rec-029] The user is working on a project called 'Heron' with a cutover date of May 30, 2026 (a Saturday
  0.654  [rec-012] The Heron project milestone schedule relative to the May 30, 2026 cutover is: Schema freeze on 
  0.602  [rec-009] The user is working on a production database migration project called Project Heron, migrating 


## 6. The memory agent (Strands)

The agent follows a MemCurrent-style design, with two automatic memory channels and one explicit channel:

- **Memory index hook**: when the agent is created, it searches memory with the user's message and appends the top-K records to the system prompt as a `<memory>` block.
- **Event capture hook**: every plain user/assistant message is written back as an AgentCore event, which feeds the next extraction. This hook is **dropped** when `read_only=True` (evaluation asks).
- **`search_memory` tool**: lets the model look for anything the index missed before it abstains.

The full-context variant has no hooks or tools; it gets the transcript in `<history>`. **Note:** Claude Sonnet 5.5 rejects an explicit `temperature`, so none is sent, and answers aren't strictly deterministic.

In [8]:
ABSTAIN = "I don't have that information."
SYSTEM_PROMPT = f"""You are a helpful personal assistant in a long-term relationship with a user.
Rules:
- Answer from what the user has told you before, as given in the context below, and from the current message.
- When facts conflict, trust the most recent one.
- If the context doesn't hold the answer to a question about the user, reply exactly "{ABSTAIN}"
- Be concise: answer in one or two sentences."""


def render_index(hits):
    body = "\n".join(f"[{rid}] {' '.join(text.split())}" for rid, _, text in hits) or "(no stored memories)"
    return f"\n\n<memory>\n{body}\n</memory>"


def render_history(sessions):
    lines = [f"[{s['started_at']}] {t['role']}: {t['text']}" for s in sessions for t in s["turns"]]
    return "\n\n<history>\n" + "\n".join(lines) + "\n</history>"

In [9]:
def build_agent(mode, *, client=None, actor_id=None, session_id=None, query="", history=(), read_only=True):
    """A fresh agent per request. mode = "memory" or "full_context"."""
    from strands import Agent, tool
    from strands.hooks import AgentInitializedEvent, HookProvider, MessageAddedEvent
    from strands.models import BedrockModel

    model = BedrockModel(model_id=MODEL_ID, region_name=REGION)  # no temperature (Sonnet 5.5)
    if mode == "full_context":
        return Agent(model=model, system_prompt=SYSTEM_PROMPT + render_history(history), callback_handler=None)

    class MemoryIndexHook(HookProvider):
        def register_hooks(self, registry, **kwargs):
            registry.add_callback(AgentInitializedEvent, self.inject)

        def inject(self, event):
            event.agent.system_prompt += render_index(search(client, MEMORY_ID, actor_id, query))

    class EventCaptureHook(HookProvider):
        def register_hooks(self, registry, **kwargs):
            registry.add_callback(MessageAddedEvent, self.capture)

        def capture(self, event):
            msg = event.message
            if any("toolUse" in b or "toolResult" in b for b in msg["content"]):
                return  # never store tool traffic as conversation
            text = "\n".join(b["text"] for b in msg["content"] if "text" in b).strip()
            if text:
                ingest_session(client, MEMORY_ID, actor_id, {
                    "session_id": session_id, "started_at": datetime.now().astimezone().isoformat(),
                    "turns": [{"role": msg["role"], "text": text}]})

    @tool
    def search_memory(query: str, k: int = 5) -> str:
        """Search long-term memory about the user. Use before saying you don't know.

        Args:
            query: What to look for, in plain words.
            k: How many records to return (1-20).
        """
        hits = search(client, MEMORY_ID, actor_id, query, max(1, min(k, 20)))
        return "\n".join(f"[{rid}] ({s:.2f}) {t}" for rid, s, t in hits) or "No matching memories."

    hooks = [MemoryIndexHook()] + ([] if read_only else [EventCaptureHook()])
    return Agent(model=model, system_prompt=SYSTEM_PROMPT, tools=[search_memory], hooks=hooks,
                 callback_handler=None)

In [10]:
import uuid

q = next(q for q in persona["questions"] if q["qid"] == "syn-sre-q24")  # a latest_value question
if RUN_LIVE:
    for mode in ("memory", "full_context"):
        agent = build_agent(mode, client=data_plane, actor_id=ACTOR, session_id=f"ask-{uuid.uuid4().hex}",
                            query=q["text"], history=persona["sessions"])
        result = agent(q["text"])
        print(f"{mode:>12}: {str(result).strip()}  (input tokens {result.metrics.accumulated_usage['inputTokens']})")
else:
    offline("ask each question in a new session under both conditions")
    rows = [{"qid": qid, "question": a["question"], "mode": m, "answer": a[m]["answer"], "tokens_in": a[m]["tokens_in"]}
            for qid, a in precomputed["example_answers"].items() for m in ("memory", "full_context")]
    table(rows, ["qid", "question", "mode", "answer", "tokens_in"])

[offline] would ask each question in a new session under both conditions


| qid | question | mode | answer | tokens_in |
|---|---|---|---|---|
| syn-sre-q05 | Which bouldering gym do I go to? | memory | You climb at Granite Yard. Your notes don't say whether it's a bouldering gym specifically, but it's the gym you go to, and it has late hours on Thursdays. | 1103 |
| syn-sre-q05 | Which bouldering gym do I go to? | full_context | Granite Yard. You mentioned going there after work, usually on Tuesday or Thursday evenings, with late hours on Thursday. | 28069 |
| syn-sre-q20 | Back in February, when was my Project Heron cutover scheduled? | memory | I don't have that information. | 5375 |
| syn-sre-q20 | Back in February, when was my Project Heron cutover scheduled? | full_context | In February, your Project Heron cutover was scheduled for April 18. You moved it to May 30 in March, and later to July 18. | 28077 |
| syn-sre-q24 | When is my Project Heron cutover currently scheduled? | memory | Your Project Heron cutover is currently scheduled for Saturday, July 18, 2026. It was originally April 18, and my notes say it was later switched to July 18. An older note also lists May 30, and the milestone dates built around it (such as Go/No-Go on May 27) may be out of date. | 5132 |
| syn-sre-q24 | When is my Project Heron cutover currently scheduled? | full_context | Your Project Heron cutover is currently scheduled for July 18, after you moved it from May 30 (and originally April 18). | 28074 |

The `syn-sre-q20` row shows the main weakness. Asked *"Back in February, when was the cutover?"*, the memory agent abstains, while full context answers April 18. Memory did retrieve a record that mentions April 18, but stored records carry no validity dates, so the agent can't tell what was true in February. Full context sees every change in order.

## 7. Build EvalKit rows

Each memory evaluator reads specific inputs. Ground truth (`relevantIds`, `newId`/`oldId`) comes from a labeling step: an LLM labeler marks which **snapshot records** state each gold fact from the timeline. Every row also has `input` and `output`.

| Evaluator | Type | Columns it reads |
|---|---|---|
| MRR, Precision@K, Recall@K | code | `retrievedIds` (ranked, top-K), `relevantIds` |
| Write Recall, Write Precision | code | `writtenFacts` (snapshot texts), `expectedFacts` (timeline statements) |
| Post-Eviction Accuracy | code | `output` (the answer), `target` (short gold span) |
| Temporal Accuracy | code | `supersedes` = `[{newId, oldId, raw}]`, where `raw` is the retrieved ranking |
| Knowledge Retention | LLM judge | `messages` (a whole conversation) |
| Summary Faithfulness | LLM judge | `context` (raw session turns); `output` is the summary |
| Contradiction Rate | LLM judge | `memories` (up to 20 stored memory texts) |

In [11]:
def retrieval_row(question, answer, retrieved_ids, relevant_ids, **tags):
    return {"input": question, "output": answer, "retrievedIds": retrieved_ids[:K], "relevantIds": relevant_ids, **tags}


def post_eviction_row(question, answer, span, **tags):
    return {"input": question, "output": answer, "target": span, **tags}


def temporal_row(question, answer, ranking, pairs, **tags):
    """pairs: [(newer_record_id, older_record_id)] for facts that superseded one another."""
    sup = [{"newId": new, "oldId": old, "raw": list(ranking)} for new, old in pairs]
    return {"input": question, "output": answer, "supersedes": sup, **tags}


ROWS = precomputed["rows"]
print({group: len(rows) for group, rows in ROWS.items()})
print(json.dumps(ROWS["retrieval"][-3], indent=1)[:900])

{'retrieval': 9, 'post_eviction': 18, 'temporal': 8, 'writes': 1, 'contradiction': 1, 'summary': 1, 'retention': 1}
{
 "input": "When is my Project Heron cutover currently scheduled?",
 "output": "Your Project Heron cutover is currently scheduled for Saturday, July 18, 2026. It was originally April 18, and my notes say it was later switched to July 18. An older note also lists May 30, and the milestone dates built around it (such as Go/No-Go on May 27) may be out of date.",
 "retrievedIds": [
  "rec-013",
  "rec-014",
  "rec-029",
  "rec-012",
  "rec-009"
 ],
 "relevantIds": [
  "rec-013"
 ],
 "qid": "syn-sre-q24",
 "case_id": "syn-sre",
 "mode": "memory",
 "category": "latest_value"
}


> **Gotcha: keep rows flat.** EvalKit's engine reads memory inputs only from a row's **top-level columns**. A nested `metadata` object is not read, so evaluators silently come back *skipped*. The reference answer goes in the **`target`** column, not `expected`.

In [12]:
def flatten(row):
    """Move nested metadata keys to top-level columns and `expected` to `target`."""
    out = {k: v for k, v in row.items() if k not in ("metadata", "expected")}
    out.update(row.get("metadata") or {})
    if "expected" in row:
        out["target"] = row["expected"]
    return out


nested = {"input": "Which gym?", "output": "Granite Yard.", "expected": "Granite Yard",
          "metadata": {"qid": "syn-sre-q05", "category": "single_fact"}}
print(flatten(nested))

{'input': 'Which gym?', 'output': 'Granite Yard.', 'qid': 'syn-sre-q05', 'category': 'single_fact', 'target': 'Granite Yard'}


## 8. Compute the code-based metrics locally

These are independent implementations written from the documented metric definitions. On the full `full-2` dataset they reproduce EvalKit's engine scores for retrieval, post-eviction and temporal accuracy. The fuzzy fact matcher uses `difflib`, while EvalKit's similarity function differs slightly, so Write Recall/Precision come out a few points lower here (0.63 / 0.30 vs the engine's 0.66 / 0.31).

In [13]:
import re
import string
from difflib import SequenceMatcher
from statistics import mean


def mrr(row):
    """1 / rank of the first relevant record (0 if none is retrieved)."""
    relevant = set(row["relevantIds"])
    return next((1 / rank for rank, rid in enumerate(row["retrievedIds"], 1) if rid in relevant), 0.0)


def precision_at_k(row, k=K):
    """Relevant records in the top k, divided by k (not by the number retrieved)."""
    return sum(rid in set(row["relevantIds"]) for rid in row["retrievedIds"][:k]) / k


def recall_at_k(row, k=K):
    """Share of all relevant records that made the top k."""
    relevant = set(row["relevantIds"])
    return len(relevant & set(row["retrievedIds"][:k])) / len(relevant)


def normalise(text):
    """SQuAD-style: lowercase, strip punctuation, drop a/an/the, collapse whitespace."""
    text = "".join(ch for ch in text.lower() if ch not in string.punctuation)
    return " ".join(re.sub(r"\b(a|an|the)\b", " ", text).split())


def post_eviction_accuracy(row):
    """1 if the normalised gold span is a substring of the normalised answer."""
    return float(normalise(row["target"]) in normalise(row["output"]))

In [14]:
def similarity(a, b):
    return SequenceMatcher(None, normalise(a), normalise(b)).ratio()


def matched_facts(written, expected, threshold=0.6):
    """One-to-one greedy matching: best-similarity pairs first, each fact used at most once."""
    pairs = sorted(((similarity(w, e), i, j) for i, w in enumerate(written) for j, e in enumerate(expected)),
                   reverse=True)
    used_w, used_e = set(), set()
    for score, i, j in pairs:
        if score < threshold:
            break
        if i not in used_w and j not in used_e:
            used_w.add(i)
            used_e.add(j)
    return len(used_e)


def write_recall(row):
    return matched_facts(row["writtenFacts"], row["expectedFacts"]) / len(row["expectedFacts"])


def write_precision(row):
    return matched_facts(row["writtenFacts"], row["expectedFacts"]) / max(len(row["writtenFacts"]), 1)


def temporal_accuracy(row):
    """Share of (newer, older) pairs where the newer record ranks above the older one in `raw`.
    A record missing from the ranking ranks last."""
    hits = []
    for p in row["supersedes"]:
        rank = {rid: i for i, rid in enumerate(p["raw"])}
        last = len(p["raw"])
        hits.append(rank.get(p["newId"], last) < rank.get(p["oldId"], last))
    return mean(hits)

In [15]:
retrieval = [{"qid": r["qid"], "category": r["category"], "MRR": round(mrr(r), 2),
              "P@5": round(precision_at_k(r), 2), "R@5": round(recall_at_k(r), 2)} for r in ROWS["retrieval"]]
table(retrieval, ["qid", "category", "MRR", "P@5", "R@5"])
print("mean  MRR %.3f  P@5 %.3f  R@5 %.3f" % tuple(mean(x[m] for x in retrieval) for m in ("MRR", "P@5", "R@5")))

| qid | category | MRR | P@5 | R@5 |
|---|---|---|---|---|
| syn-sre-q01 | single_fact | 1.0 | 0.2 | 1.0 |
| syn-sre-q02 | single_fact | 0.5 | 0.4 | 0.29 |
| syn-sre-q05 | single_fact | 1.0 | 0.6 | 1.0 |
| syn-sre-q07 | single_fact | 1.0 | 0.2 | 1.0 |
| syn-sre-q04 | single_fact | 1.0 | 0.2 | 1.0 |
| syn-sre-q20 | temporal | 1.0 | 0.2 | 1.0 |
| syn-sre-q24 | latest_value | 1.0 | 0.2 | 1.0 |
| syn-sre-q30 | procedural | 1.0 | 0.8 | 0.8 |
| syn-sre-q31 | procedural | 1.0 | 0.8 | 0.8 |

mean  MRR 0.944  P@5 0.400  R@5 0.877


In [16]:
pe = {}
for r in ROWS["post_eviction"]:
    pe.setdefault(r["qid"], {"qid": r["qid"], "target": r["target"]})[r["mode"]] = post_eviction_accuracy(r)
table(list(pe.values()), ["qid", "target", "memory", "full_context"])
for mode in ("memory", "full_context"):
    print(f"Post-Eviction Accuracy ({mode}): {mean(v[mode] for v in pe.values()):.3f}")

w = ROWS["writes"][0]
print(f"\nWrite Recall {write_recall(w):.3f}  Write Precision {write_precision(w):.3f}  "
      f"({len(w['writtenFacts'])} stored records vs {len(w['expectedFacts'])} expected facts)")
t = [temporal_accuracy(r) for r in ROWS["temporal"]]
print(f"Temporal Accuracy over {len(t)} questions about changed facts: {mean(t):.3f}")

| qid | target | memory | full_context |
|---|---|---|---|
| syn-sre-q01 | Quillfield Payments | 1.0 | 1.0 |
| syn-sre-q02 | Project Heron | 1.0 | 1.0 |
| syn-sre-q05 | Granite Yard | 1.0 | 1.0 |
| syn-sre-q07 | 99.95 percent | 0.0 | 1.0 |
| syn-sre-q04 | Biscuit | 1.0 | 1.0 |
| syn-sre-q20 | April 18 | 0.0 | 1.0 |
| syn-sre-q24 | July 18 | 1.0 | 1.0 |
| syn-sre-q30 | incident commander | 1.0 | 1.0 |
| syn-sre-q31 | every thirty minutes | 1.0 | 1.0 |

Post-Eviction Accuracy (memory): 0.778
Post-Eviction Accuracy (full_context): 1.000



Write Recall 0.308  Write Precision 0.055  (145 stored records vs 26 expected facts)
Temporal Accuracy over 8 questions about changed facts: 0.438


These scores cover only the 9 sample questions, so they differ from the full-run numbers in section 11. Precision@5 is low by design: most questions have one or two relevant records, and the formula divides by 5. Write Precision is low because AgentCore stores the same fact several times, as a semantic fact, a preference and part of a summary, and only one copy can match each expected fact. Persona `sre` is a hard case for write metrics: 145 long, merged records against 26 short expected facts.

## 9. LLM-judge metrics

The three judge evaluators break their input into items and ask one verdict question per item:

| Evaluator | Item | Verdict | Score |
|---|---|---|---|
| Knowledge Retention | each fact the user supplied in `messages` | yes / no: kept without being re-asked, forgotten or contradicted? | fraction yes (no facts = 1) |
| Summary Faithfulness | each atomic claim in the summary | yes / no / idk: supported by the `context` turns? | fraction yes |
| Contradiction Rate | each pair of `memories` (≤ 20, so ≤ 190 pairs) | do the two make incompatible claims about one subject? | flagged pairs ÷ all pairs (**lower is better**) |

Below is a compact judge built on the Bedrock Converse API. The prompts are paraphrased for this cookbook. EvalKit's managed evaluators use their own prompts and judge model, so treat local scores as estimates.

In [17]:
JUDGE_PROMPTS = {
    "knowledge_retention": (
        "You check whether an assistant keeps what a user tells it. Find every fact the user supplied "
        "(preferences, identifiers, constraints, decisions). For each, decide if the assistant later "
        "asked for it again, behaved as if it forgot it, or contradicted it.",
        'Reply with JSON only: {"items": [{"fact": "...", "verdict": "yes|no", "reason": "..."}]} '
        "where yes means the fact was retained."),
    "summary_faithfulness": (
        "You check a conversation summary against the original messages. Split the summary into atomic "
        "claims. A claim with a changed number, date, name or decision is not supported.",
        'Reply with JSON only: {"items": [{"claim": "...", "verdict": "yes|no|idk", "reason": "..."}]} '
        "(yes = supported, no = contradicted, idk = neither)."),
    "contradiction_rate": (
        "You check a numbered list of stored memories for internal conflicts. Two memories conflict only "
        "if they make incompatible claims about the same subject; extra detail is not a conflict.",
        'Reply with JSON only, listing conflicting pairs by number: {"pairs": [{"a": 1, "b": 4, "reason": "..."}]}'),
}


def bedrock_judge(model_id=MODEL_ID):
    import boto3

    rt = boto3.client("bedrock-runtime", region_name=REGION)

    def call(system, prompt):
        resp = rt.converse(modelId=model_id, system=[{"text": system}],
                           messages=[{"role": "user", "content": [{"text": prompt}]}],
                           inferenceConfig={"maxTokens": 4096})
        return "".join(b.get("text", "") for b in resp["output"]["message"]["content"])

    return call


def ask_json(judge, name, body):
    system, task = JUDGE_PROMPTS[name]
    raw = judge(system, f"{body}\n\n{task}")
    return json.loads(re.search(r"\{.*\}", raw, re.S).group(0))

In [18]:
def knowledge_retention(row, judge):
    convo = "\n".join(f"{m['role']}: {m['content']}" for m in row["messages"])
    items = ask_json(judge, "knowledge_retention", f"Conversation:\n{convo}")["items"]
    return 1.0 if not items else mean(i["verdict"] == "yes" for i in items)


def summary_faithfulness(row, judge):
    body = "Original messages:\n" + "\n".join(row["context"]) + f"\n\nSummary:\n{row['output']}"
    items = ask_json(judge, "summary_faithfulness", body)["items"]
    return mean(i["verdict"] == "yes" for i in items) if items else None


def contradiction_rate(row, judge):
    memories = row["memories"][:20]
    listing = "\n".join(f"[{i}] {m}" for i, m in enumerate(memories, 1))
    pairs = {tuple(sorted((p["a"], p["b"]))) for p in ask_json(judge, "contradiction_rate", f"Stored memories:\n{listing}")["pairs"]}
    n = len(memories)
    return len(pairs) / (n * (n - 1) // 2)

Offline, a **replay judge** returns the verdicts recorded in run `full-2`, so the scoring code runs unchanged. Live, the same functions call Bedrock.

In [19]:
V = precomputed["judge_verdicts"]
replay = {
    "knowledge_retention": lambda s, p: json.dumps({"items": V["knowledge_retention"]["items"]}),
    "summary_faithfulness": lambda s, p: json.dumps({"items": V["summary_faithfulness"]["items"]}),
    "contradiction_rate": lambda s, p: json.dumps({"pairs": [{"a": a, "b": b} for a, b in V["contradiction_rate"]["flagged"]]}),
}
scorers = {"knowledge_retention": (knowledge_retention, "retention"),
           "summary_faithfulness": (summary_faithfulness, "summary"),
           "contradiction_rate": (contradiction_rate, "contradiction")}
for name, (fn, group) in scorers.items():
    judge = bedrock_judge() if RUN_LIVE else replay[name]
    print(f"{name:22s} {fn(ROWS[group][0], judge):.3f}   ({ROWS[group][0].get('case_id')})")

knowledge_retention    0.000   (rt-projects-01)
summary_faithfulness   0.857   (syn-sre)
contradiction_rate     0.068   (syn-sre)


In [20]:
sf = V["summary_faithfulness"]["items"]
print(f"Summary of {V['summary_faithfulness']['session']}: {sum(i['verdict'] == 'yes' for i in sf)}/{len(sf)} claims supported")
for i in sf:
    if i["verdict"] != "yes":
        print(f"  [{i['verdict']}] {i['claim'][:110]}\n         {i['reason'][:110]}")

mems = ROWS["contradiction"][0]["memories"]
a, b = V["contradiction_rate"]["flagged"][0]
print(f"\nContradiction example (1 of {len(V['contradiction_rate']['flagged'])} flagged pairs):"
      f"\n  [{a}] {mems[a - 1][:110]}\n  [{b}] {mems[b - 1][:110]}")

print("\nKnowledge Retention, rt-projects-01 (a custom retention scenario that fails in both conditions):")
for i in V["knowledge_retention"]["items"][:2]:
    print(f"  [{i['verdict']}] {i['fact']}")

Summary of syn-sre-s08: 12/14 claims supported
  [no] Schema freeze is Fri Apr 25, which is 5 weeks / 35 days before cutover.
         The conversation established that Apr 25 is a Saturday, so 'Fri Apr 25' is wrong. The final agreed date was Fr
  [no] Apr 24 is only noted as an alternative to avoid the weekend.
         The assistant's corrected final list set the schema freeze at Fri Apr 24, and the user accepted it. It was the

Contradiction example (1 of 13 flagged pairs):
  [1] The user lives in Brindlemoor, where their team is also based.
  [7] The user is moving home with one other person from a flat, with approximately 40 boxes to pack. They have rece

Knowledge Retention, rt-projects-01 (a custom retention scenario that fails in both conditions):
  [no] Ember board demo is on 14 October
  [no] The user's emissions-reporting user story topic (context for 'also suggest acceptance criteria')


## 10. Send to EvalKit with the SDK

There are two ways to get data into EvalKit, and they feed different evaluators:

- **Dataset rows → code metrics.** `evalkit.Eval(name, data=rows, evaluators=[...])` scores the flat rows from section 7. Code metrics need the gold labels (`relevantIds`, `target`, `supersedes`, ...), which only rows carry.
- **Traces → LLM judges in the UI.** Upload each conversation as a trace under one agent name (here `Memory-eval`), one session per user and condition. The judges read the conversation and retrieved contexts from the trace.

Evaluator **rule ids are org-scoped**: copy them from the EvalKit UI (Evaluators → open a managed memory evaluator → copy its id) into `RULE_IDS`.

In [21]:
RULE_IDS = {  # placeholders: paste your org's evaluator ids
    "retrieval": ["<MRR_RULE_ID>", "<PRECISION_AT_K_RULE_ID>", "<RECALL_AT_K_RULE_ID>"],
    "writes": ["<WRITE_RECALL_RULE_ID>", "<WRITE_PRECISION_RULE_ID>"],
    "post_eviction": ["<POST_EVICTION_ACCURACY_RULE_ID>"],
    "temporal": ["<TEMPORAL_ACCURACY_RULE_ID>"],
    "retention": ["<KNOWLEDGE_RETENTION_RULE_ID>"],
    "summary": ["<SUMMARY_FAITHFULNESS_RULE_ID>"],
    "contradiction": ["<CONTRADICTION_RATE_RULE_ID>"],
}

if RUN_LIVE:
    import evalkit

    ek = evalkit.init(subscription_key=os.environ["EVALKIT_SUBSCRIPTION_KEY"], service_name="agentcore-memory-eval",
                      environment=os.environ.get("EVALKIT_ENVIRONMENT", "development"), function_tracing=False)
    collections = [c for c in [os.environ.get("EVALKIT_COLLECTION_ID")] if c]
    for group, rows in ROWS.items():
        result = evalkit.Eval(f"cookbook-{group}", data=rows, evaluators=RULE_IDS[group],
                              collections=collections, judge_model=os.environ.get("EVALKIT_JUDGE_MODEL"))
        print(group, result.overall_scores)
else:
    for group, rows in ROWS.items():
        offline(f"run evalkit.Eval('cookbook-{group}') on {len(rows)} rows with {len(RULE_IDS[group])} evaluator(s)")

[offline] would run evalkit.Eval('cookbook-retrieval') on 9 rows with 3 evaluator(s)
[offline] would run evalkit.Eval('cookbook-post_eviction') on 18 rows with 1 evaluator(s)
[offline] would run evalkit.Eval('cookbook-temporal') on 8 rows with 1 evaluator(s)
[offline] would run evalkit.Eval('cookbook-writes') on 1 rows with 2 evaluator(s)
[offline] would run evalkit.Eval('cookbook-contradiction') on 1 rows with 1 evaluator(s)
[offline] would run evalkit.Eval('cookbook-summary') on 1 rows with 1 evaluator(s)
[offline] would run evalkit.Eval('cookbook-retention') on 1 rows with 1 evaluator(s)


In [22]:
import secrets
from datetime import timezone

AGENT_NAME = "Memory-eval"


def upload_answer(client, *, session_id, user_id, app_version, question, answer, retrieved_texts, attrs=None):
    """One trace per question: a root span plus a retrieval span the judges can read."""
    from evalkit import TraceEnvelope

    trace_id, now = secrets.token_hex(16), datetime.now(timezone.utc).isoformat()
    common = dict(trace_id=trace_id, created_at=now, start_time=now, end_time=now, agent_name=AGENT_NAME,
                  session_id=session_id, user_id=user_id, app_version=app_version, service_name="agentcore-memory-eval")
    root = secrets.token_hex(8)
    client.enqueue(TraceEnvelope(span_id=root, span_type="function_call", operation="run_agent", **common,
                                 attributes={"function.args": json.dumps([question]), "function.result": answer,
                                             "gen_ai.agent.name": AGENT_NAME, **(attrs or {})}))
    client.enqueue(TraceEnvelope(span_id=secrets.token_hex(8), parent_span_id=root, span_type="db_query",
                                 operation="search_memory", **common,
                                 attributes={"evalkit.retrieved_contexts": json.dumps(retrieved_texts)}))


if RUN_LIVE:
    for qid, a in precomputed["example_answers"].items():
        for mode in ("memory", "full_context"):
            upload_answer(ek, session_id=f"cookbook/syn-sre/{mode}", user_id="syn-sre", app_version=f"cookbook-{mode}",
                          question=a["question"], answer=a[mode]["answer"],
                          retrieved_texts=[h["text"] for h in a[mode]["retrieved"]])
    print("flushed:", evalkit.flush())
else:
    offline(f"upload {2 * len(precomputed['example_answers'])} traces under agent {AGENT_NAME!r}, "
            "one session per condition, then run the LLM judges from the EvalKit UI")

[offline] would upload 6 traces under agent 'Memory-eval', one session per condition, then run the LLM judges from the EvalKit UI


## 11. Results from the full run (`full-2`)

The full run covered 30 memory users (5 synthetic personas and 25 custom scenarios), 245 questions and 490 answers with 0 errors. Code metrics were computed with EvalKit's engine. The LLM-judge numbers are a local Claude replica of EvalKit's judges.

In [23]:
summary = json.loads((DATA / "full2_summary.json").read_text())
table([{**m, "full_context": m["full_context"] if m["full_context"] is not None else "n/a"} for m in summary["metrics"]],
      ["family", "metric", "type", "memory", "full_context", "pass", "n"])
eff = summary["efficiency"]
table([
    {"measure": "Input tokens per question", "memory": eff["input_tokens_per_question"]["memory"],
     "full_context": eff["input_tokens_per_question"]["full_context"], "ratio": f'{eff["input_tokens_per_question"]["ratio"]:.2f}x'},
    {"measure": "Cost per conversation (USD)", "memory": eff["cost_per_conversation_usd"]["memory"],
     "full_context": eff["cost_per_conversation_usd"]["full_context"], "ratio": f'{eff["cost_per_conversation_usd"]["ratio"]:.2f}x'},
    {"measure": "False abstention", "memory": f'{eff["false_abstention"]["memory"]:.0%}',
     "full_context": f'{eff["false_abstention"]["full_context"]:.0%}', "ratio": ""},
], ["measure", "memory", "full_context", "ratio"])
table(summary["weak_slices_post_eviction"], ["slice", "memory", "full_context"])

| family | metric | type | memory | full_context | pass | n |
|---|---|---|---|---|---|---|
| Accurate retrieval | MRR | code | 0.798 | n/a | >= 0.50 | 192 |
| Accurate retrieval | Precision@5 | code | 0.394 | n/a | >= 0.50 | 192 |
| Accurate retrieval | Recall@5 | code | 0.788 | n/a | >= 0.50 | 192 |
| Test-time learning | Write Recall | code | 0.663 | n/a | >= 0.50 | 30 |
| Test-time learning | Write Precision | code | 0.314 | n/a | >= 0.50 | 30 |
| Test-time learning | Knowledge Retention | llm | 0.8 | 0.87 | >= 0.50 | 5 |
| Long-range | Post-Eviction Accuracy | code | 0.805 | 0.945 | = 1 | 200 |
| Long-range | Summary Faithfulness | llm | 0.96 | n/a | >= 0.50 | 179 |
| Conflict resolution | Temporal Accuracy | code | 0.664 | n/a | >= 0.50 | 61 |
| Conflict resolution | Contradiction Rate (lower is better) | llm | 0.08 | n/a | <= 0.10 | 10 |

| measure | memory | full_context | ratio |
|---|---|---|---|
| Input tokens per question | 2412 | 24763 | 0.10x |
| Cost per conversation (USD) | 0.096 | 0.412 | 0.23x |
| False abstention | 7% | 3% |  |

| slice | memory | full_context |
|---|---|---|
| synthetic / temporal | 0.57 | 0.9 |
| workflows (procedural) | 0.73 | 0.88 |

### Findings

1. **Memory is a strong trade-off.** It keeps about 85% of the ceiling's accuracy (0.805 vs 0.945) with 0.10× the input tokens at 0.23× the cost.
2. **Retrieval works.** The relevant record is in the top 5 for 79% of questions (Recall@5) and is usually ranked first (MRR 0.80).
3. **Temporal memory is the main weakness** (0.57 vs 0.90). Superseded values and their dates are dropped or merged, so "what was it in February?" can't be answered.
4. **Procedural memory lags** (0.73 vs 0.88). Ordered multi-step procedures are split into separate facts.
5. **Stale facts linger.** Temporal Accuracy is 0.66 and Contradiction Rate is 0.08: old and new values are both stored, and the old one sometimes ranks first.

### Next steps

- Keep superseded values with their validity dates (a custom strategy or a self-managed history namespace), then re-run and compare Temporal Accuracy.
- Add a procedural namespace that stores ordered steps plus their trigger as one record.
- Re-rank by recency, and deduplicate across strategies to raise Write Precision and Precision@5.
- Run the three LLM judges in the EvalKit UI for official scores.

## Cleanup

Delete the memory when you're done. This removes all of its events and records.

In [24]:
if RUN_LIVE:
    import boto3

    boto3.client("bedrock-agentcore-control", region_name=REGION).delete_memory(memoryId=MEMORY_ID)
    print("memory deleted")
else:
    offline("delete the memory and all its events and records")

[offline] would delete the memory and all its events and records
